In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from sklearn.metrics import (roc_auc_score, classification_report,
                              average_precision_score, precision_recall_curve,
                              confusion_matrix)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('../models', exist_ok=True)
os.makedirs('../plots', exist_ok=True)

# Load clean data
df = pd.read_csv('../data/fraud_clean.csv')
FEATURES = [col for col in df.columns if col != 'Class']
X = df[FEATURES]
y = df['Class']

print(f"Features: {len(FEATURES)}")
print(f"Samples:  {len(X):,}")
print(f"Fraud:    {y.sum():,} ({y.mean()*100:.3f}%)")

# Stratified split — maintains fraud rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.30,
    random_state = 42,
    stratify     = y    # CRITICAL for imbalanced data
)
print()
print(f"Train: {X_train.shape[0]:,} | Fraud: {y_train.mean()*100:.3f}%")
print(f"Test:  {X_test.shape[0]:,}  | Fraud: {y_test.mean()*100:.3f}%")

Features: 35
Samples:  19,786
Fraud:    32 (0.162%)

Train: 13,850 | Fraud: 0.159%
Test:  5,936  | Fraud: 0.168%


In [3]:
print("BEFORE SMOTE:")
print(f"  Legitimate: {(y_train==0).sum():,}")
print(f"  Fraud:      {(y_train==1).sum():,}")
print(f"  Ratio: {(y_train==0).sum()/(y_train==1).sum():.0f}:1")

# Apply SMOTE ONLY on training data — NEVER on test data
sm = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

print()
print("AFTER SMOTE:")
print(f"  Legitimate: {(y_train_sm==0).sum():,}")
print(f"  Fraud:      {(y_train_sm==1).sum():,}")
print(f"  Ratio: 1:1 — Balanced!")
print()
print("Test set UNCHANGED (real-world imbalanced distribution maintained)")

BEFORE SMOTE:
  Legitimate: 13,828
  Fraud:      22
  Ratio: 629:1

AFTER SMOTE:
  Legitimate: 13,828
  Fraud:      13,828
  Ratio: 1:1 — Balanced!

Test set UNCHANGED (real-world imbalanced distribution maintained)


In [4]:
# Calculate class weight ratio for scale_pos_weight
neg_pos_ratio = (y_train_sm==0).sum() / (y_train_sm==1).sum()

xgb = XGBClassifier(
    n_estimators       = 300,
    max_depth          = 6,
    learning_rate      = 0.05,
    subsample          = 0.8,
    colsample_bytree   = 0.8,
    scale_pos_weight   = neg_pos_ratio,  # handles remaining imbalance
    eval_metric        = 'auc',
    early_stopping_rounds = 20,
    random_state       = 42,
    n_jobs             = -1,
    verbosity          = 0              # Fix: removed use_label_encoder (deprecated)
)

print("Training XGBoost...")
xgb.fit(
    X_train_sm, y_train_sm,
    eval_set   = [(X_test, y_test)],
    verbose    = 50
)

# Predictions
y_pred_proba = xgb.predict_proba(X_test)[:, 1]
y_pred       = xgb.predict(X_test)

# Metrics
auc_roc = roc_auc_score(y_test, y_pred_proba)
auc_pr  = average_precision_score(y_test, y_pred_proba)

print()
print("=" * 50)
print(f"AUC-ROC:  {auc_roc:.4f}   (target > 0.95)")
print(f"AUC-PR:   {auc_pr:.4f}    (better for imbalanced)")
print()
print("CLASSIFICATION REPORT (default threshold 0.5):")
print(classification_report(y_test, y_pred, target_names=['Legitimate','Fraud']))

Training XGBoost...
[0]	validation_0-auc:0.52055
[28]	validation_0-auc:0.54607

AUC-ROC:  0.5937   (target > 0.95)
AUC-PR:   0.0039    (better for imbalanced)

CLASSIFICATION REPORT (default threshold 0.5):
              precision    recall  f1-score   support

  Legitimate       1.00      0.81      0.89      5926
       Fraud       0.00      0.20      0.00        10

    accuracy                           0.81      5936
   macro avg       0.50      0.50      0.45      5936
weighted avg       1.00      0.81      0.89      5936



In [5]:
# Find optimal threshold using Precision-Recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Find threshold where precision >= 80% AND recall is maximized
valid_idx = np.where(precision[:-1] >= 0.80)[0]
if len(valid_idx) > 0:
    best_idx          = valid_idx[np.argmax(recall[valid_idx])]
    optimal_threshold = float(thresholds[best_idx])
else:
    optimal_threshold = 0.5   # fallback

print(f"Default threshold:  0.5000")
print(f"Optimal threshold:  {optimal_threshold:.4f}")
print()

# Compare default vs optimal
y_pred_default = (y_pred_proba >= 0.5).astype(int)
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)

caught_def = (y_pred_default[y_test == 1] == 1).sum()
false_def  = (y_pred_default[y_test == 0] == 1).sum()
caught_opt = (y_pred_optimal[y_test == 1] == 1).sum()
false_opt  = (y_pred_optimal[y_test == 0] == 1).sum()
total_fraud = (y_test == 1).sum()

print(f"Default (0.5):  Caught {caught_def}/{total_fraud} fraud | {false_def} false alarms")
print(f"Optimal ({optimal_threshold:.3f}): Caught {caught_opt}/{total_fraud} fraud | {false_opt} false alarms")
print()
print(f"Extra fraud caught:   +{caught_opt - caught_def}")
print(f"Extra false alarms:   +{false_opt - false_def}")
print(f"Net value: ${(caught_opt-caught_def)*500 - (false_opt-false_def)*1:,} (each fraud=$500, each FP=$1)")

# Threshold tuning plot
plt.figure(figsize=(10, 5))
plt.plot(thresholds, precision[:-1], label='Precision', color='blue',  linewidth=2)
plt.plot(thresholds, recall[:-1],    label='Recall',    color='red',   linewidth=2)
plt.axvline(optimal_threshold, color='green', linestyle='--',
            linewidth=2, label=f'Optimal = {optimal_threshold:.3f}')
plt.axvline(0.5, color='gray', linestyle=':', linewidth=1.5, label='Default (0.5)')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Threshold Tuning — Precision vs Recall', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../plots/threshold_tuning.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: plots/threshold_tuning.png")

Default threshold:  0.5000
Optimal threshold:  0.5000

Default (0.5):  Caught 2/10 fraud | 1136 false alarms
Optimal (0.500): Caught 2/10 fraud | 1136 false alarms

Extra fraud caught:   +0
Extra false alarms:   +0
Net value: $0 (each fraud=$500, each FP=$1)
Saved: plots/threshold_tuning.png


In [7]:
import shap

print("Computing SHAP values (1-2 minutes)...")
explainer   = shap.TreeExplainer(xgb)
X_sample    = X_test.iloc[:1000]   # sample for speed
shap_values = explainer.shap_values(X_sample)

# Global feature importance
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    'Feature': FEATURES,
    'Mean_SHAP': mean_abs_shap
}).sort_values('Mean_SHAP', ascending=False)

print("TOP 10 FRAUD INDICATORS (SHAP importance):")
print(shap_df.head(10).to_string(index=False))

# SHAP bar chart
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample,
                   feature_names=FEATURES,
                   plot_type='bar', show=False)
plt.title('Feature Importance — SHAP', fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/shap_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: plots/shap_importance.png")

# SHAP beeswarm (direction of impact)
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_sample,
                   feature_names=FEATURES,
                   plot_type='dot', show=False, max_display=15)
plt.title('SHAP Beeswarm — Impact Direction', fontweight='bold')
plt.tight_layout()
plt.savefig('../plots/shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: plots/shap_beeswarm.png")

# Individual explanation for 1 fraud case
fraud_indices = np.where((y_test.values == 1) & (y_pred_proba > 0.7))[0]
if len(fraud_indices) > 0:
    idx = fraud_indices[0]
    single_shap = explainer.shap_values(X_test.iloc[[idx]])
    print()
    print(f"TRANSACTION {idx} — ACTUAL: FRAUD")
    print(f"Model confidence: {y_pred_proba[idx]*100:.1f}%")
    print("TOP 5 REASONS THIS WAS FLAGGED:")
    feat_contribs = list(zip(FEATURES, single_shap[0]))
    feat_contribs.sort(key=lambda x: abs(x[1]), reverse=True)
    for feat, val in feat_contribs[:5]:
        d = 'INCREASES' if val > 0 else 'DECREASES'
        print(f"  {feat}: {d} fraud risk by {abs(val):.4f}")

Computing SHAP values (1-2 minutes)...
TOP 10 FRAUD INDICATORS (SHAP importance):
   Feature  Mean_SHAP
Is_Morning   0.150818
       V12   0.139400
        V7   0.110473
Amount_Log   0.076742
       V23   0.039234
  Is_Night   0.036424
       V22   0.035439
       V21   0.031340
       V25   0.028579
    Amount   0.026020
Saved: plots/shap_importance.png
Saved: plots/shap_beeswarm.png


In [8]:
import joblib
import json
import os
os.makedirs('../models', exist_ok=True)

# Save model
joblib.dump(xgb, '../models/fraud_model.pkl')
print("Saved: models/fraud_model.pkl")

# Save feature names (API uses exact same order)
with open('../models/feature_names.json', 'w') as f:
    json.dump(FEATURES, f)
print("Saved: models/feature_names.json")

# Save optimal threshold
with open('../models/optimal_threshold.json', 'w') as f:
    json.dump({'threshold': optimal_threshold}, f)
print("Saved: models/optimal_threshold.json")

# Save all metrics
caught_opt = (y_pred_optimal[y_test == 1] == 1).sum()
false_opt  = (y_pred_optimal[y_test == 0] == 1).sum()
metrics = {
    'auc_roc':              round(float(auc_roc), 4),
    'auc_pr':               round(float(auc_pr), 4),
    'optimal_threshold':    round(optimal_threshold, 4),
    'fraud_caught':         int(caught_opt),
    'total_fraud':          int((y_test==1).sum()),
    'false_positives':      int(false_opt),
    'false_positive_rate':  round(false_opt / (y_test==0).sum(), 4),
    'training_samples':     int(len(X_train_sm)),
    'test_samples':         int(len(X_test))
}
with open('../models/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Saved: models/metrics.json")

print()
print("ALL MODELS/ FILES:")
for fname in os.listdir('../models'):
    size = os.path.getsize(f'../models/{fname}') / 1024
    print(f"  {fname}: {size:.1f} KB")
print()
print(f"Final AUC-ROC:  {auc_roc:.4f}")
print(f"Fraud caught:   {caught_opt}/{(y_test==1).sum()} ({caught_opt/(y_test==1).sum()*100:.1f}%)")
print(f"False positive: {false_opt} ({false_opt/(y_test==0).sum()*100:.2f}%)")

Saved: models/fraud_model.pkl
Saved: models/feature_names.json
Saved: models/optimal_threshold.json
Saved: models/metrics.json

ALL MODELS/ FILES:
  feature_names.json: 0.3 KB
  fraud_model.pkl: 81.4 KB
  metrics.json: 0.2 KB
  optimal_threshold.json: 0.0 KB

Final AUC-ROC:  0.5937
Fraud caught:   2/10 (20.0%)
False positive: 1136 (19.17%)
